In [55]:
import pandas as pd
from rdkit import Chem
def get_reactive_atom_indices(smiles):
    # 解析 SMILES
    mol = Chem.MolFromSmiles(smiles)
    
    # 获取标记为反应位点的原子
    reactive_atoms = []
    for atom in mol.GetAtoms():
        # 检查是否带有反应位点标记
        if atom.HasProp('molAtomMapNumber'):
            reactive_atoms.append(atom.GetIdx())  # 获取原子序号
    
    return reactive_atoms

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_smiles_with_atom_map(smiles: str) -> str:
    try:
        # 将 SMILES 转换为分子对象
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("Invalid SMILES string.")
        
        # 提取位点信息（Atom Maps）
        atom_map = {atom.GetIdx(): atom.GetAtomMapNum() for atom in mol.GetAtoms() if atom.GetAtomMapNum() > 0}
        
        # 标准化分子（使用 Standardizer）
        uncharger = rdMolStandardize.Uncharger()  # 去质子化
        mol = uncharger.uncharge(mol)
        
        # 去除多余氢原子
        mol = Chem.RemoveHs(mol)
        
        # 恢复位点信息（Atom Maps）
        for idx, map_num in atom_map.items():
            mol.GetAtomWithIdx(idx).SetAtomMapNum(map_num)
        
        # 返回标准化后的 SMILES
        standardized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        return standardized_smiles
    except Exception as e:
        return f"Error: {e}"

def df_col_getatomindices(dfname,col_old,flat=False):
    preds = dfname[col_old].to_list()
    preds = [i.split('|') for i in preds]
    try:
        preds = [list(map(standardize_smiles_with_atom_map,i)) for i in preds]
        ranks = [list(map(get_reactive_atom_indices,i)) for i in preds]
    except:
        ranks = []
        for k in preds:
            preds_part = []
            for x in k:
                try:
                    x = standardize_smiles_with_atom_map(x)
                    x_new = get_reactive_atom_indices(x)
                except:
                    x_new = []
                preds_part.append(x_new)
            ranks.append(preds_part)
    if flat == True:
        ranks_new = []
        for i in ranks:
            list_part = []
            for j in i:
                list_part = list_part + j
            ranks_new.append(list_part)
        return ranks_new
    else:
        return ranks
    
def canonicalsmiles(smi):
    mol = Chem.MolFromSmiles(smi)
    smi_new = Chem.MolToSmiles(mol)
    return smi_new

In [56]:
df = pd.read_csv('all_123.csv')
substrate = df['substrate'].to_list()
substrate = list(map(canonicalsmiles,substrate))
sites_true = df_col_getatomindices(df,'site_truth',flat=True)
metapredictor_predict = df_col_getatomindices(df,'metapredictor_predict',flat=False)
our_predict = df_col_getatomindices(df,'our_predict',flat=False)
df['substrate'] = substrate
df['sites'] = sites_true
df['metapredictor_ranks'] = metapredictor_predict
df['ourmodel_ranks'] = our_predict
df

[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Running Uncharger
[11:18:35] Run

,substrate,metapredictor_predict,site_truth,site_count,our_predict,sites,metapredictor_ranks,ourmodel_ranks
0,c1cnc(N2CCNCC2)nc1,c1nc(N2CCNCC2)nc[cH:1]1|c1nc(N2CCNCC2)n[cH:1]c...,c1nc(N2CCNCC2)nc[cH:1]1,1,[CH:1]1=CN=C(N2CCNCC2)N=C1|C1=CN=C(N2CCN[CH2:1...,[11],"[[11], [2], [8], [7], [2, 11], [3, 5, 6, 7], [...","[[11], [8], [7], [7], [9], [8], [9], [7], [2],..."
1,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,CC(/C=C/[C:1]12O[C:1]1(C)CCCC2(C)C)=C\C=C\C(C)...,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\[C:1]...,1,C12(/C=C/C(C)=C/C=C/C(C)=C/C([OH:1])=O)OC1(C)C...,"[20, 22]","[[4, 6], [22], [9], [10], [13], [4, 11, 13], [...","[[22], [4], [4], [8], [10], [9], [22], [22], [..."
2,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,CC(=O)[NH:1]c1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2...,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=N[CH2:1]C(=O)N2,1,CC(=O)NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=N[CH2:1]...,[19],"[[3], [19], [14], [20, 21, 22], [20, 22], [9, ...","[[19], [18], [14], [20], [19], [13], [15], [18..."
3,CC(=O)Nc1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc4ccc(...,CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,2,CC(=O)NC1=[CH:1]C=CC(N2C(=O)N(C3CC3)C(=O)C3=C(...,"[3, 2, 3, 36]","[[3], [29], [20], [24], [30], [34], [26], [15]...","[[36], [27], [28], [34], [7], [34], [35], [23]..."
4,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,1,CC(C)([CH3:1])C1=CC(C(C)(C)C)=C(NC(=O)C2=CNC3=...,[10],"[[12], [7], [26], [12], [17], [7, 11, 12, 27, ...","[[10], [10], [12], [21], [12], [20], [23], [13..."
...,...,...,...,...,...,...,...,...
118,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccccc3)(...,OC(c1ccccc1)(c1ccccc1)C12CC[N+:1](CCOCc3ccccc3...,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3cc[cH:1]...,2,OC(C1=CC=CC=C1)(C1=CC=CC=C1)C12CC[N+](CCO[CH2:...,"[25, 20]","[[17], [20], [17, 20], [11], [25], [11], [19],...","[[21], [17], [11], [31], [25], [30], [12], [21..."
119,OC(Cl)(Cl)C(c1ccc(Cl)cc1)c1ccccc1Cl,O[C:1](Cl)(Cl)C(c1ccc(Cl)cc1)c1ccccc1Cl|Clc1cc...,Clc1ccc(C(c2ccccc2Cl)[C:1](Cl)(Cl)[OH:1])cc1,1,OC(Cl)(Cl)C(C1=CC=C(Cl)C=C1)C1=CC=[CH:1]C=C1Cl...,"[13, 16]","[[1], [16], [15], [4], [18], [1, 18], [1, 4], ...","[[15], [14], [15], [4], [16], [14], [1], [10],..."
120,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4cc[cH:1]c...,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccc([OH:...,1,OC1=CC=C([CH2:1]OCC[N+]23CCC(C(O)(C4=CC=CC=C4)...,[26],"[[24], [30], [9], [26], [24], [6], [32], [26],...","[[5], [32], [30], [26], [26], [9], [24], [31],..."
121,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2,Fc1ccccc1C1=NCc2cnc(C[OH:1])n2-c2ccc(Cl)cc21|O...,Fc1ccccc1C1=NCc2cnc(C[OH:1])n2-c2ccc(Cl)cc21|O...,3,OCC1=NC=C2N1C1=CC=C(Cl)C=C1[C:1](C1=CC=CC=C1F)...,"[15, 6, 23]","[[15], [1], [14, 15], [6], [15, 20], [9, 15], ...","[[14], [14, 23], [15], [23], [14], [14, 23], [..."


In [54]:
# dict_sub2site = dict(zip(substrate,sites_true))
# import pickle

# # 文件路径
# file_path = "dict_sub2site.pickle"

# # 将字典保存到pickle文件
# with open(file_path, 'wb') as file:
#     pickle.dump(dict_sub2site, file)

# print(f"字典已保存到 {file_path}")


字典已保存到 dict_sub2site.pickle


In [57]:
dict_sub2sitetrue = dict(zip(substrate,sites_true))
dict_sub2metarank = dict(zip(substrate,metapredictor_predict))
print(len(list(dict_sub2sitetrue.keys())))

123


In [58]:
df = pd.read_csv('predict_site_123_top6_seed_2024.csv')
substrate = df['substrate'].to_list()
substrate = list(map(canonicalsmiles,substrate))
df['substrate'] = substrate
sites_true = [dict_sub2sitetrue[i] for i in substrate]
metapredictor_predict = [dict_sub2metarank[i] for i in substrate]
df['site_truth'] = sites_true
# sites_true = df_col_getatomindices(df,'site_truth',flat=True)
# metapredictor_predict = df_col_getatomindices(df,'metapredictor_predict',flat=False)
our_predict = df_col_getatomindices(df,'pred_site',flat=False)
df['sites'] = sites_true
df['metapredictor_ranks'] = metapredictor_predict
df['ourmodel_ranks'] = our_predict
df

[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Running Uncharger
[11:18:49] Run

,Unnamed: 0,substrate,predict,pred_site,site_truth,sites,metapredictor_ranks,ourmodel_ranks
0,0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\[C:1]...,"[20, 22]","[20, 22]","[[4, 6], [22], [9], [10], [13], [4, 11, 13], [...","[[20, 22], [11, 13], [20, 22], [10], [20, 22],..."
1,1,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2|CC(=...,CC(=O)[NH:1]c1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2...,[19],[19],"[[3], [19], [14], [20, 21, 22], [20, 22], [9, ...","[[3], [19], [18], [3, 10, 18], [3, 10, 18, 19]..."
2,2,CC(=O)Nc1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc4ccc(...,CC1=C2C(=C(NC3=CC=C(I)C=C3F)N(C)C1=O)C(=O)N(C1...,CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,"[3, 2, 3, 36]","[3, 2, 3, 36]","[[3], [29], [20], [24], [30], [34], [26], [15]...","[[3], [36], [29], [7], [28], [34], [23], [35],..."
3,3,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)C1=CC(C(C)(C)CO)=C(NC(=O)C2=CNC3=CC=CC...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(NC(=O)c2c[nH]c3c...,[10],[10],"[[12], [7], [26], [12], [17], [7, 11, 12, 27, ...","[[10], [11, 12], [10], [12], [21], [20], [11, ..."
4,4,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CC(C)(CO)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],[39],"[[39], [32], [7], [1, 39], [27], [39], [27, 32...","[[39], [30, 31], [26, 30], [32], [24], [4, 5, ..."
...,...,...,...,...,...,...,...,...
118,118,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccccc3)(...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CC[O:1]Cc3cccc...,"[25, 20]","[25, 20]","[[17], [20], [17, 20], [11], [25], [11], [19],...","[[20], [17], [25], [11], [25], [30], [31], [18..."
119,119,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc(OCCOC4CC4)cc...,O=C(O)C1OC(O[C@@H]2[C@@H](O)[C@H](C3=CC=C(Cl)C...,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc(OCCOC4CC4)cc...,"[28, 29, 19, 18, 19, 11]","[28, 29, 19, 18, 19, 11]","[[27], [29], [29], [19], [16], [28, 29], [18, ...","[[28, 29], [16], [19], [22], [24], [18, 19], [..."
120,120,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2,O=CC1=CN=C(CO)N1C1=CC=C(Cl)C=C1C(=O)C1=CC=CC=C...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=N[CH2:1]2|...,"[15, 6, 23]","[15, 6, 23]","[[15], [1], [14, 15], [6], [15, 20], [9, 15], ...","[[23], [14, 15], [2], [23], [15], [1], [2, 5, ..."
121,121,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,Oc1ccc(C[O:1]CC[N+]23CCC(C(O)(c4ccccc4)c4ccccc...,[26],[26],"[[24], [30], [9], [26], [24], [6], [32], [26],...","[[6], [1, 17, 26], [30], [32], [10, 11, 12, 13..."


In [41]:
df = pd.read_csv('predict_site_123_top10.csv')
substrate = df['substrate'].to_list()
substrate = list(map(canonicalsmiles,substrate))
df['substrate'] = substrate
sites_true = [dict_sub2sitetrue[i] for i in substrate]
metapredictor_predict = [dict_sub2metarank[i] for i in substrate]
df['site_truth'] = sites_true
# sites_true = df_col_getatomindices(df,'site_truth',flat=True)
# metapredictor_predict = df_col_getatomindices(df,'metapredictor_predict',flat=False)
our_predict = df_col_getatomindices(df,'pred_site',flat=False)
df['sites'] = sites_true
df['metapredictor_ranks'] = metapredictor_predict
df['ourmodel_ranks'] = our_predict
df

[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Running Uncharger
[14:32:27] Run

,name,substrate,predict,pred_site,site_truth,sites,metapredictor_ranks,ourmodel_ranks
0,0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\[C:1]...,"[20, 22]","[20, 22]","[[4, 6], [22], [9], [10], [13], [4, 11, 13], [...","[[20, 22], [11, 13], [11, 13], [8], [10], [], ..."
1,1,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2|CC(=...,CC(=O)[NH:1]c1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2...,[19],[19],"[[3], [19], [14], [20, 21, 22], [20, 22], [9, ...","[[3], [19], [18], [14], [20, 22], [10, 18, 19]..."
2,2,CC(=O)Nc1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc4ccc(...,CC1=C2C(=C(NC3=CC=C(I)C=C3F)N(C)C1=O)C(=O)N(C1...,CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,"[3, 2, 3, 36]","[3, 2, 3, 36]","[[3], [29], [20], [24], [30], [34], [26], [15]...","[[3], [36], [], [28], [29], [34], [7], [34], [..."
3,3,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)C1=CC(C(C)(C)CO)=C(O)C=C1NC(=O)C1=CNC2...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,[10],[10],"[[12], [7], [26], [12], [17], [7, 11, 12, 27, ...","[[10], [10], [11, 12], [21], [12], [20], [11, ..."
4,4,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CC(C)(CO)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],[39],"[[39], [32], [7], [1, 39], [27], [39], [27, 32...","[[39], [32], [26, 30], [30, 31], [4, 5, 6, 38,..."
...,...,...,...,...,...,...,...,...
118,118,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccccc3)(...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CC[O:1]Cc3cccc...,"[25, 20]","[25, 20]","[[17], [20], [17, 20], [11], [25], [11], [19],...","[[20], [17], [11], [25], [31], [30], [30, 31],..."
119,119,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc(OCCOC4CC4)cc...,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(O)C=C3)=...,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc([O:1]CCOC4CC...,"[28, 29, 19, 18, 19, 11]","[28, 29, 19, 18, 19, 11]","[[27], [29], [29], [19], [16], [28, 29], [18, ...","[[16], [22], [28, 29], [18, 19], [24], [19], [..."
120,120,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2,O=CC1=CN=C(CO)N1C1=CC=C(Cl)C=C1C(=O)C1=CC=CC=C...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=N[CH2:1]2|...,"[15, 6, 23]","[15, 6, 23]","[[15], [1], [14, 15], [6], [15, 20], [9, 15], ...","[[23], [2], [15], [22, 23], [2, 3, 4, 5, 6], [..."
121,121,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,Oc1ccc(C[O:1]CC[N+]23CCC(C(O)(c4ccccc4)c4ccccc...,[26],[26],"[[24], [30], [9], [26], [24], [6], [32], [26],...","[[6], [32], [30], [26], [26], [1, 17, 26], [10..."


In [27]:
df = pd.read_csv('123_site_from_smiles.csv')
substrate = df['substrate'].to_list()
substrate = list(map(canonicalsmiles,substrate))
df['substrate'] = substrate
sites_true = [dict_sub2sitetrue[i] for i in substrate]
metapredictor_predict = [dict_sub2metarank[i] for i in substrate]
df['site_truth'] = sites_true
# sites_true = df_col_getatomindices(df,'site_truth',flat=True)
# metapredictor_predict = df_col_getatomindices(df,'metapredictor_predict',flat=False)
our_predict = df_col_getatomindices(df,'pred_site',flat=False)
df['sites'] = sites_true
df['metapredictor_ranks'] = metapredictor_predict
df['ourmodel_ranks'] = our_predict
df

[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Running Uncharger
[13:55:01] Run

,substrate,predict,metabolite,site_truth,pred_site,sites,metapredictor_ranks,ourmodel_ranks
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,CC(/C=C/[C@]12O[C@]1(C)CCCC2(C)C)=C\C=C\C(C)=C...,"[20, 22]",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\[C:1]...,"[20, 22]","[[4, 6], [22], [9], [10], [13], [4, 11, 13], [...","[[20, 22], [11, 13], [11, 13], [8], [10], [9],..."
1,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2|CC(=...,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NC(O)C(=O)N2,[19],CC(=O)[NH:1]c1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2...,[19],"[[3], [19], [14], [20, 21, 22], [20, 22], [9, ...","[[3], [19], [18], [14], [20, 22], [10, 18, 19]..."
2,CC(=O)Nc1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc4ccc(...,CC1=C2C(=C(NC3=CC=C(I)C=C3F)N(C)C1=O)C(=O)N(C1...,Cc1c(=O)n(C)c(Nc2ccc(I)cc2F)c2c(=O)n(C3CC3)c(=...,"[3, 2, 3, 36]",CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,"[3, 2, 3, 36]","[[3], [29], [20], [24], [30], [34], [26], [15]...","[[3], [36], [], [28], [29], [34], [7], [34], [..."
3,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)C1=CC(C(C)(C)CO)=C(O)C=C1NC(=O)C1=CNC2...,CC(C)(C)c1cc(C(C)(C)C(=O)O)c(O)cc1NC(=O)c1c[nH...,[10],CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,[10],"[[12], [7], [26], [12], [17], [7, 11, 12, 27, ...","[[10], [10], [11, 12], [21], [12], [20], [23],..."
4,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CC(C)(CO)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=...,CC(C)(CO)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OC...,[39],CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],"[[39], [32], [7], [1, 39], [27], [39], [27, 32...","[[39], [32], [26, 30], [30, 31], [4, 5, 6, 38,..."
...,...,...,...,...,...,...,...,...
118,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccccc3)(...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,OCC[N+]12CCC(C(O)(c3ccccc3)c3ccccc3)(CC1)CC2|O...,"[25, 20]",OC(c1ccccc1)(c1ccccc1)C12CC[N+](CC[O:1]Cc3cccc...,"[25, 20]","[[17], [20], [17, 20], [11], [25], [11], [19],...","[[20], [17], [11], [31], [25], [30], [12], [21..."
119,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc(OCCOC4CC4)cc...,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(O)C=C3)=...,O=C(O)[C@H]1OC(O[C@@H]2[C@@H](O)[C@H](c3ccc(Cl...,"[28, 29, 19, 18, 19, 11]",OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc([O:1]CCOC4CC...,"[28, 29, 19, 18, 19, 11]","[[27], [29], [29], [19], [16], [28, 29], [18, ...","[[16], [22], [28, 29], [18, 19], [24], [19], [..."
120,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2,O=CC1=CN=C(CO)N1C1=CC=C(Cl)C=C1C(=O)C1=CC=CC=C...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2O|O=C(O...,"[15, 6, 23]",OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=N[CH2:1]2|...,"[15, 6, 23]","[[15], [1], [14, 15], [6], [15, 20], [9, 15], ...","[[23], [2], [15], [14, 15], [22, 23], [2, 3, 4..."
121,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,O=C(O)C1OC(Oc2ccc(COCC[N+]34CCC(C(O)(c5ccccc5)...,[26],Oc1ccc(C[O:1]CC[N+]23CCC(C(O)(c4ccccc4)c4ccccc...,[26],"[[24], [30], [9], [26], [24], [6], [32], [26],...","[[6], [32], [30], [26], [26], [1, 17, 26], [10..."


In [49]:
def get_unique_numbers(nums, n):
    """
    从列表中提取前 n 个不同的数字。

    参数:
        nums (list): 输入的数字列表。
        n (int): 需要提取的不同数字的数量。

    返回:
        list: 包含前 n 个不同数字的列表。
    """
    unique_nums = []
    seen = set()

    for num in nums:
        if num not in seen:
            unique_nums.append(num)
            seen.add(num)
        if len(unique_nums) == n:
            break

    return unique_nums

In [50]:
metapredictor_results_old = df['metapredictor_ranks'].to_list()
ourmodel_results_old = df['ourmodel_ranks'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)
metapredictor_results_new = []
for i in metapredictor_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    metapredictor_results_new.append(list_part)

true_sites = df['sites'].to_list()

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

metapredictor_top1_acc = acc_top_n_old(1,metapredictor_results_new,true_sites)
metapredictor_top3_acc = acc_top_n_old(3,metapredictor_results_new,true_sites)
metapredictor_top5_acc = acc_top_n_old(5,metapredictor_results_new,true_sites)
metapredictor_topn = []
metapredictor_topn.append(metapredictor_top1_acc)
metapredictor_topn.append(metapredictor_top3_acc)
metapredictor_topn.append(metapredictor_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

top 1 accuracy = 0.5609756097560976
top 3 accuracy = 0.8373983739837398
top 5 accuracy = 0.8861788617886179
top 1 accuracy = 0.5284552845528455
top 3 accuracy = 0.8048780487804879
top 5 accuracy = 0.9105691056910569


In [51]:
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['metapredictor_topn'] = metapredictor_topn
df['ourmodel_topn'] = ourmodel_topn
df

,top_n,metapredictor_topn,ourmodel_topn
0,top1_recall,0.560976,0.528455
1,top3_recall,0.837398,0.804878
2,top5_recall,0.886179,0.910569
